# シーン検索デモを Colab で起動する（A100）

2 段構成（VideoJev-4B → 学習済み Qwen3-VL-4B）+ 3 段の振り分け。起動すると共有リンク（gradio.live）が出るので、ブラウザで開く。

前提: Drive の `MyDrive/movie_llm/grpo3_time/step_00400/`（LoRA）と、`videos_2fps_s360.zip`（サンプルに使う縮小済み動画）。

In [ ]:
%cd /content
!rm -rf movie_llm && git clone -q https://github.com/kanaharu20/movie_llm.git
%cd /content/movie_llm
!pip install -q "transformers==5.14.1" "accelerate>=1.10" "peft>=0.15" "einops>=0.8" "scipy>=1.12" "av>=14" "pillow>=10" "pandas>=2" "pyarrow" "huggingface_hub" "gradio>=5"
!pip install -q "flash-linear-attention==0.4.2" "fla-core==0.4.2" || echo 'fla なしで続行'
!pip uninstall -y -q torchao
!pip install -q -e . --no-deps

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/movie_llm'
import os, shutil, zipfile
os.makedirs('/content/samples', exist_ok=True)
# サンプル: Drive の movie_llm/samples/（CC BY の日本の動画 3 本、縮小済み）。無ければベンチマークの縮小動画から 3 本
import glob
src = glob.glob(f'{DRIVE}/samples/*.mp4')
if src:
    for p in src:
        shutil.copy(p, '/content/samples/')
else:
    with zipfile.ZipFile(f'{DRIVE}/videos_2fps_s360.zip') as z:
        for i in sorted(z.infolist(), key=lambda i: i.file_size)[:3]:
            with z.open(i) as s, open(f'/content/samples/{os.path.basename(i.filename)}', 'wb') as d:
                shutil.copyfileobj(s, d)
print('samples', os.listdir('/content/samples'))

In [ ]:
%cd /content/movie_llm
import os
os.environ['MOVIE_LLM_SAMPLES'] = '/content/samples'
!python -m movie_llm.demo.app --adapter {DRIVE}/grpo3_time/step_00400 --samples /content/samples --max-frames 2048 --share

出力の `Running on public URL: https://xxxx.gradio.live` を開く。初回はモデルの読み込みで 1〜2 分、1 回の検索は 10〜20 秒。